In [ ]:
# import torch
# import numpy as np
# import random
# from datasets import load_dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForSeq2SeqLM,
#     Seq2SeqTrainer,
#     Seq2SeqTrainingArguments,
#     DataCollatorForSeq2Seq,
# )
# from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# MODEL_NAME = "google/flan-t5-small"
# MODEL_NAME = "google/flan-t5-large"
MODEL_NAME = "google/flan-t5-base"
DATASET_NAME = "databricks/databricks-dolly-15k"
SUBSET_SIZE = 1000
SEED = 42
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128
OUTPUT_DIR = "./output_random_1k"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME, split="train")
dataset = dataset.shuffle(seed=SEED).select(range(SUBSET_SIZE))

def format_example(example):
    # Combine instruction + context (if present) into a single input
    if example.get("context"):
        prompt = f"Instruction: {example['instruction']}\nContext: {example['context']}"
    else:
        prompt = f"Instruction: {example['instruction']}"
    return {"input_text": prompt, "target_text": example["response"]}

dataset = dataset.map(format_example)

# Train/eval split (90/10)
split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

Loading dataset...
Train size: 900, Eval size: 100


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=example["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess, remove_columns=eval_dataset.column_names)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# !pip install -U torchao -q

In [ ]:
print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],  # standard for T5 attention
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [ ]:
# Set these two per model:
PER_DEVICE_TRAIN_BS = 3
GRAD_ACCUM_STEPS = 2
PER_DEVICE_EVAL_BS = 2   # eval doesn't need to match train's effective size, just needs to fit memory

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
    max_steps=300,        # <-- fixed number of optimizer updates, same for every model
    # num_train_epochs=0.5,          # matches the 0.5-epoch setting used elsewhere
    # num_train_epochs=1.0,      #large
    learning_rate=3e-3,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none",
    predict_with_generate=True,
    fp16=False,   # T5 unstable in fp16 on T4 - keep disabled, see earlier NaN issue
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
# import torch
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [ ]:
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss
1,2.306348,1.227386
2,2.816867,1.201863


TrainOutput(global_step=300, training_loss=4.590271288553874, metrics={'train_runtime': 156.1458, 'train_samples_per_second': 11.528, 'train_steps_per_second': 1.921, 'total_flos': 618727735296000.0, 'train_loss': 4.590271288553874, 'epoch': 2.0})

In [ ]:
print("Evaluating...")
eval_results = trainer.evaluate()
print(eval_results)
print(f"\nFinal eval loss: {eval_results.get('eval_loss')}")
print("If this ran without errors and produced a numeric eval_loss, the pipeline works on free tier.")

Evaluating...


Training Loss,Validation Loss,Epoch
2.816867,1.201863,2


{'eval_loss': 1.201863169670105}

Final eval loss: 1.201863169670105
If this ran without errors and produced a numeric eval_loss, the pipeline works on free tier.


In [ ]:
############### Without Accumulation ###########################
# print("Starting training...")
# trainer.train()

# # --- Cell 8: Evaluate ---
# print("Evaluating...")
# eval_results = trainer.evaluate()
# print(eval_results)

# # --- Cell 9: Save eval loss for comparison ---
# print(f"\nFinal eval loss: {eval_results.get('eval_loss')}")
# print("If this ran without errors and produced a numeric eval_loss, the pipeline works on free tier.")